In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    return attn_weights @ V, attn_weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model, self.num_heads = d_model, num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch, seq_len, _ = x.size()
        return x.view(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch, _, seq_len, _ = x.size()
        x = x.transpose(1, 2).contiguous()
        return x.view(batch, seq_len, self.d_model)

    def forward(self, query, key, value, mask=None):
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))
        if mask is not None:
            mask = mask.unsqueeze(1)
        out, attn = scaled_dot_product_attention(Q, K, V, mask)
        return self.W_o(self.combine_heads(out)), attn

In [2]:
class PositionalEncoding(nn.Module):
    """给每个位置一个身份标签, 让 attention 能区分顺序"""

    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        # 等价于 1 / 10000^(2i/d), 用 exp-log 改写是为了防溢出
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)   # 偶数维放 sin
        pe[:, 1::2] = torch.cos(position * div_term)   # 奇数维放 cos

        # buffer: 不可学习, 但 model.to(device) 时会跟着走
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # 按序列实际长度取前几行
        return x + self.pe[:, :x.size(1)]


class FeedForward(nn.Module):
    """逐位置独立加工: 升维 -> 非线性 -> 降回原维"""

    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # nn.Linear 只作用在最后一维, 所以天然"逐位置独立"
        return self.linear2(F.relu(self.linear1(x)))


class TransformerBlock(nn.Module):
    """两个子层, 每个都是 x = x + Sublayer(LayerNorm(x))  [Pre-LN]"""

    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.ln1 = nn.LayerNorm(d_model)   # 两个 LN 各有独立的 gamma/beta,
        self.ln2 = nn.LayerNorm(d_model)   # 不能复用同一个

    def forward(self, x, mask=None):
        # 子层一: 横向, token 之间交流
        h = self.ln1(x)                          # 只清洗读进子层的这一份
        attn_out, attn_w = self.attn(h, h, h, mask)
        x = x + attn_out                         # 加回原始 x, 旁路保持干净

        # 子层二: 纵向, 每个位置独立加工
        x = x + self.ff(self.ln2(x))
        return x, attn_w

In [3]:
d_model, num_heads, d_ff = 512, 8, 2048
block = TransformerBlock(d_model, num_heads, d_ff)
x = torch.randn(2, 5, d_model)

out, attn = block(x)
print("输入形状:", x.shape)
print("输出形状:", out.shape, "  <- 必须完全一样, 这是能堆叠的前提")
print("权重形状:", attn.shape)

# 参数量分解
n_attn = sum(p.numel() for p in block.attn.parameters())
n_ff   = sum(p.numel() for p in block.ff.parameters())
print(f"\nAttention: {n_attn:,}")
print(f"FFN      : {n_ff:,}   <- 约为 attention 的 {n_ff/n_attn:.1f} 倍")

# 位置编码长什么样
pe_small = PositionalEncoding(d_model=64, max_len=100)
table = pe_small.pe[0]
print(f"\n编码表形状: {tuple(table.shape)}")
print(f"数值范围  : {table.min():.3f} ~ {table.max():.3f}   <- 永远在 [-1,1]")
print("位置  0   :", [round(v, 3) for v in table[0, :8].tolist()])
print("位置  1   :", [round(v, 3) for v in table[1, :8].tolist()])
print("位置 50   :", [round(v, 3) for v in table[50, :8].tolist()])

# 堆 6 层
pos_enc = PositionalEncoding(d_model)
blocks = nn.ModuleList([
    TransformerBlock(d_model, num_heads, d_ff) for _ in range(6)
])
h = pos_enc(x)
for blk in blocks:
    h, _ = blk(h)
print("\n堆完 6 层后:", h.shape, "  <- 形状始终不变")

输入形状: torch.Size([2, 5, 512])
输出形状: torch.Size([2, 5, 512])   <- 必须完全一样, 这是能堆叠的前提
权重形状: torch.Size([2, 8, 5, 5])

Attention: 1,050,624
FFN      : 2,099,712   <- 约为 attention 的 2.0 倍

编码表形状: (100, 64)
数值范围  : -1.000 ~ 1.000   <- 永远在 [-1,1]
位置  0   : [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
位置  1   : [0.841, 0.54, 0.682, 0.732, 0.533, 0.846, 0.409, 0.912]
位置 50   : [-0.262, 0.965, -0.203, 0.979, 0.157, -0.988, 0.787, -0.617]

堆完 6 层后: torch.Size([2, 5, 512])   <- 形状始终不变


In [4]:
x = torch.randn(1, 4, d_model)
perm = torch.tensor([2, 0, 3, 1])          # 一个打乱顺序
blk = TransformerBlock(d_model, num_heads, d_ff)

# 不加位置编码
a, _ = blk(x)
b, _ = blk(x[:, perm, :])
print("不加位置编码 —— 打乱输入的输出 == 原输出的打乱?")
print("  ", torch.allclose(b, a[:, perm, :], atol=1e-5))

# 加位置编码
c, _ = blk(pos_enc(x))
d, _ = blk(pos_enc(x[:, perm, :]))
print("加了位置编码 —— 同样的检查?")
print("  ", torch.allclose(d, c[:, perm, :], atol=1e-5))

不加位置编码 —— 打乱输入的输出 == 原输出的打乱?
   True
加了位置编码 —— 同样的检查?
   False
